# 準備演習 03: 構造化データ抽出パイプラインの構築

## 目的

- Claude Agent SDK で structured extraction を実装する
- JSON schema で nullable / optional を適切に設計する
- semantic validation (calculated_total vs stated_total) を実装する
- エラーコンテキスト付き validation-retry ループを実装する
- 情報が存在しない時に `null` を返し、捏造しない設計を体験する

## 対象ドメイン

- Domain 4: Prompt Engineering & Structured Output
- Domain 5: Context Management & Reliability

## 完成イメージ

この Notebook では **Claude Agent SDK の実際の structured extraction** を体験します。
最新の best practice は公式ドキュメントと完成版 Lab を参照してください:

- 公式: `https://platform.claude.com/docs/en/agent-sdk/overview`
- 完成版 Lab: [../labs/03-structured-extraction/](../labs/03-structured-extraction/)

In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from typing import Any
from dotenv import load_dotenv

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient, create_sdk_mcp_server, tool
from claude_agent_sdk.types import AssistantMessage, ResultMessage, TextBlock, ToolUseBlock

load_dotenv()

MAX_RETRIES = 3

def section(title: str):
    print(f"\n=== {title} ===")

## Step 1. JSON Schema を設計する

構造化抽出のための JSON Schema を定義します。nullable フィールドと required フィールドを適切に分離します。

In [ ]:
INVOICE_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "invoice_number": {
            "type": "string",
            "description": "請求書番号 (例: INV-2024-001)",
        },
        "vendor_name": {
            "type": "string",
            "description": "請求元企業名",
        },
        "invoice_date": {
            "type": ["string", "null"],
            "description": "請求日 YYYY-MM-DD 形式。読み取れない場合は null",
        },
        "due_date": {
            "type": ["string", "null"],
            "description": "支払期限 YYYY-MM-DD 形式。記載がない場合は null",
        },
        "line_items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "quantity": {"type": "number"},
                    "unit_price": {"type": "number"},
                    "amount": {
                        "type": "number",
                        "description": "quantity * unit_price と一致する金額",
                    },
                },
                "required": ["description", "quantity", "unit_price", "amount"],
            },
        },
        "tax_amount": {
            "type": ["number", "null"],
            "description": "消費税額。記載がない場合は null",
        },
        "stated_total": {
            "type": "number",
            "description": "請求書に記載された合計金額をそのまま転記",
        },
        "calculated_total": {
            "type": "number",
            "description": "line_items.amount の合計 + tax_amount から計算した合計金額",
        },
        "discount_amount": {
            "type": "number",
            "description": "割引額 (割引がある場合のみ)",
        },
        "payment_method": {
            "type": "string",
            "enum": ["bank_transfer", "credit_card", "cash", "other"],
            "description": "支払い方法",
        },
        "payment_method_detail": {
            "type": ["string", "null"],
            "description": "payment_method が other の場合の詳細",
        },
    },
    "required": [
        "invoice_number",
        "vendor_name",
        "line_items",
        "stated_total",
        "calculated_total",
        "payment_method",
    ],
}

section("JSON Schema 定義")
print("Required フィールド:")
for field in INVOICE_SCHEMA["required"]:
    print(f"  - {field}")
print("\nNullable フィールド:")
for field, spec in INVOICE_SCHEMA["properties"].items():
    if isinstance(spec.get("type"), list) and "null" in spec["type"]:
        print(f"  - {field}")

### 確認ポイント

- **required**: 常に存在すべきフィールド
- **nullable**: キーは存在するが値は `null` を許容
- **optional**: 情報がない場合はキーごと省略できる (discount_amount など)

## Step 2. サンプル請求書データを準備する

In [ ]:
SAMPLE_INVOICE_GOOD = """
請求書番号: INV-2024-001
請求元: Acme Supplies株式会社
請求日: 2024-01-15
支払期限: 2024-02-15

品目:
1. キーボード x 2個 @ 5,000円 = 10,000円
2. マウス x 3個 @ 2,000円 = 6,000円

小計: 16,000円
消費税 (10%): 1,600円
合計: 17,600円

支払方法: 銀行振込
""".strip()

SAMPLE_INVOICE_INCOMPLETE = """
請求書番号: INV-2024-002
請求元: Beta Corporation

品目:
1. モニター x 2個 @ 15,000円 = 30,000円

合計: 33,000円

支払方法: その他 (PayPal)
""".strip()

SAMPLE_INVOICE_ERRORS = """
請求書番号: INV-2024-003
請求元: Gamma Tech
請求日: 2024-03-01

品目:
1. ラップトップ x 1個 @ 80,000円 = 80,000円
2. アダプター x 2個 @ 3,000円 = 5,000円  # 計算エラー: 正しくは 6,000円

小計: 86,000円
消費税 (10%): 8,600円
合計: 95,000円  # エラー: 正しくは 94,600円

支払方法: クレジットカード
""".strip()

section("サンプル請求書データ")
print("✓ SAMPLE_INVOICE_GOOD (完全なデータ)")
print("✓ SAMPLE_INVOICE_INCOMPLETE (一部データ欠損)")
print("✓ SAMPLE_INVOICE_ERRORS (計算エラーあり)")

## Step 3. Validation ロジックを実装する

In [ ]:
@dataclass
class ValidationResult:
    is_valid: bool
    errors: list[str]


def validate_invoice(data: dict[str, Any]) -> ValidationResult:
    """請求書データを検証する"""
    errors = []
    
    # Required フィールドのチェック
    for field in INVOICE_SCHEMA["required"]:
        if field not in data:
            errors.append(f"必須フィールドが欠損: {field}")
    
    # line_items の計算チェック
    for i, item in enumerate(data.get("line_items", [])):
        expected = item["quantity"] * item["unit_price"]
        actual = item["amount"]
        if not math.isclose(actual, expected, abs_tol=0.01):
            errors.append(
                f"明細 {i+1} ({item['description']}): "
                f"金額不一致 (期待値: {expected}円, 実際: {actual}円)"
            )
    
    # calculated_total の検証
    if "line_items" in data and "calculated_total" in data:
        items_total = sum(item["amount"] for item in data["line_items"])
        tax = data.get("tax_amount") or 0
        discount = data.get("discount_amount") or 0
        expected_total = items_total + tax - discount
        actual_total = data["calculated_total"]
        
        if not math.isclose(actual_total, expected_total, abs_tol=0.01):
            errors.append(
                f"calculated_total 不一致 "
                f"(期待値: {expected_total}円, 実際: {actual_total}円)"
            )
    
    # stated_total vs calculated_total の比較
    if "stated_total" in data and "calculated_total" in data:
        if not math.isclose(data["stated_total"], data["calculated_total"], abs_tol=0.01):
            errors.append(
                f"stated_total と calculated_total が不一致 "
                f"(記載: {data['stated_total']}円, 計算: {data['calculated_total']}円)"
            )
    
    return ValidationResult(is_valid=len(errors) == 0, errors=errors)


section("Validation ロジック")
print("✓ validate_invoice 関数")
print("  - Required フィールドチェック")
print("  - line_items 計算チェック")
print("  - calculated_total 検証")
print("  - stated_total vs calculated_total 比較")

## Step 4. Claude Agent SDK ツールを定義する

`@tool` デコレーターで構造化抽出ツールを定義します。

In [ ]:
@tool(
    "extract_invoice",
    "請求書テキストから構造化データを抽出します。欠損情報は null を使って返してください。捏造は厳禁です。",
    INVOICE_SCHEMA,
)
async def extract_invoice_tool(args: dict[str, Any]) -> dict[str, Any]:
    """請求書抽出ツール (実際の抽出は Claude が行う)"""
    # このツールは schema を定義するだけで、実際の抽出は Claude が行う
    return {
        "content": [
            {"type": "text", "text": "Structured invoice extraction received for validation."}
        ]
    }


section("Agent SDK ツール定義")
print("✓ @tool extract_invoice")
print("  - JSON Schema に基づく構造化出力")
print("  - nullable フィールドのサポート")

## Step 5. 抽出と Validation を実行する関数を実装する

In [ ]:
async def extract_invoice(
    invoice_text: str,
    verbose: bool = True,
) -> dict[str, Any]:
    """請求書テキストから構造化データを抽出する"""
    # MCP サーバーを作成
    server = create_sdk_mcp_server(
        name="extraction",
        version="1.0.0",
        tools=[extract_invoice_tool],
    )
    
    # プロンプトを構築
    prompt = f"""
以下の請求書テキストから構造化データを抽出してください。

## 重要なルール
1. 読み取れない情報は null を使用してください
2. 情報を捏造しないでください
3. line_items の amount は quantity * unit_price と一致する必要があります
4. calculated_total は line_items の合計 + tax_amount で計算してください
5. stated_total は請求書に記載された値をそのまま転記してください

請求書テキスト:
{invoice_text}
""".strip()
    
    # エージェントオプション
    options = ClaudeAgentOptions(
        max_turns=4,
        mcp_servers={"extraction": server},
        output_format={
            "type": "json_schema",
            "schema": INVOICE_SCHEMA,
        },
    )
    
    extracted_data = None
    
    # エージェントを実行
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock) and verbose:
                        print(f"\n📝 Claude: {block.text[:100]}...")
                    elif isinstance(block, ToolUseBlock) and verbose:
                        print(f"\n🔧 ツール使用: {block.name}")
            elif isinstance(message, ResultMessage):
                if message.structured_output:
                    extracted_data = message.structured_output
    
    if not extracted_data:
        raise RuntimeError("構造化出力が返されませんでした")
    
    return extracted_data


async def extract_with_retry(
    invoice_text: str,
    verbose: bool = True,
) -> tuple[dict[str, Any], ValidationResult]:
    """Validation と Retry を含む抽出パイプライン"""
    last_data = None
    last_validation = None
    
    for attempt in range(1, MAX_RETRIES + 1):
        if verbose:
            print(f"\n{'='*60}")
            print(f"試行 {attempt}/{MAX_RETRIES}")
            print(f"{'='*60}")
        
        # 抽出を実行
        if attempt == 1:
            extracted_data = await extract_invoice(invoice_text, verbose=verbose)
        else:
            # リトライ時は前回のエラーをフィードバック
            retry_prompt = f"""
前回の抽出に以下のエラーがありました:
{chr(10).join('- ' + e for e in last_validation.errors)}

同じ schema を維持したまま、これらのエラーを修正してください。
修正できない項目は null を使用してください。

元の請求書テキスト:
{invoice_text}
""".strip()
            extracted_data = await extract_invoice(retry_prompt, verbose=verbose)
        
        last_data = extracted_data
        
        # Validation を実行
        validation = validate_invoice(extracted_data)
        last_validation = validation
        
        if verbose:
            print(f"\n📊 Validation 結果:")
            if validation.is_valid:
                print("✅ 検証成功")
            else:
                print(f"❌ 検証失敗 ({len(validation.errors)} 件のエラー)")
                for error in validation.errors:
                    print(f"  - {error}")
        
        if validation.is_valid:
            return extracted_data, validation
    
    # MAX_RETRIES に達しても成功しなかった
    if verbose:
        print(f"\n⚠️ {MAX_RETRIES} 回の試行後も検証に失敗しました")
    
    return last_data, last_validation


section("抽出・検証関数")
print("✓ extract_invoice (Claude Agent SDK を使用)")
print("✓ extract_with_retry (Validation と Retry ループ)")

## Step 6. シナリオ1: 完全なデータ

In [ ]:
section("シナリオ1: 完全なデータ")
print(SAMPLE_INVOICE_GOOD)

result, validation = await extract_with_retry(SAMPLE_INVOICE_GOOD, verbose=True)

print("\n📋 抽出結果:")
print(json.dumps(result, ensure_ascii=False, indent=2))

### 確認ポイント

1. すべての required フィールドが正しく抽出されたか
2. line_items の計算が正しいか
3. stated_total と calculated_total が一致するか
4. 1回の試行で検証に成功したか

## Step 7. シナリオ2: 不完全なデータ (nullable フィールド)

In [ ]:
section("シナリオ2: 不完全なデータ (nullable フィールド)")
print(SAMPLE_INVOICE_INCOMPLETE)

result, validation = await extract_with_retry(SAMPLE_INVOICE_INCOMPLETE, verbose=True)

print("\n📋 抽出結果:")
print(json.dumps(result, ensure_ascii=False, indent=2))

print("\n🔍 Nullable フィールドの確認:")
for field in ["invoice_date", "due_date", "tax_amount"]:
    value = result.get(field)
    print(f"  {field}: {value} ({'null' if value is None else 'set'})")

### 確認ポイント

1. 欠損している情報は `null` として抽出されたか
2. payment_method="other" の場合、payment_method_detail が設定されているか
3. 情報を捏造せずに、存在しない情報は null にできているか

## Step 8. シナリオ3: 計算エラーあり (Retry ループ)

In [ ]:
section("シナリオ3: 計算エラーあり (Retry ループ)")
print(SAMPLE_INVOICE_ERRORS)

result, validation = await extract_with_retry(SAMPLE_INVOICE_ERRORS, verbose=True)

print("\n📋 最終抽出結果:")
print(json.dumps(result, ensure_ascii=False, indent=2))

if validation.is_valid:
    print("\n✅ Claude が計算エラーを修正しました")
else:
    print(f"\n⚠️ 修正不可能なエラーが残っています: {validation.errors}")

### 確認ポイント

1. Claude が元のテキストの計算エラーを検出したか
2. Retry ループで正しい金額に修正されたか
3. stated_total (請求書の記載値) と calculated_total (正しい計算値) の差異が明確になっているか

## まとめ

この演習では、以下を学びました:

1. **JSON Schema 設計**: nullable, required, optional の使い分け
2. **@tool デコレーター**: 構造化抽出ツールの定義
3. **output_format**: Agent SDK の structured output 機能
4. **Semantic Validation**: 計算ロジックに基づく検証
5. **Validation-Retry ループ**: エラーフィードバックによる改善
6. **Null vs Fabrication**: 存在しない情報を捏造せず null を返す設計

## 完成版 Lab 参照

より詳細な実装とバッチ処理、human review routing については、完成版 Lab を参照してください:
- [../labs/03-structured-extraction/](../labs/03-structured-extraction/)
- Claude Agent SDK 公式ドキュメント: https://platform.claude.com/docs/en/agent-sdk/overview